# Privacy-Preserving On-Device LLMs for Activity of Daily Living (ADL) Recognition
### Full implementation of the manuscript pipeline

This notebook implements every component described in the manuscript, in the same order as Algorithm 1:

| Notebook section | Manuscript section | What it does |
|---|---|---|
| 1. Setup & configuration | 3.5 | Models, Ollama endpoint, seeds, window parameters |
| 2. Data loading | 3.2 | UCI ADL (Home A / B) and MARBLE loaders (+ synthetic demo data) |
| 3. Windowing & textualization | 3.4 | 60 s / 16 s windows, 80 % overlap, max-overlap labels, label-leak check |
| 4. Partitioning | 3.3 | Instance-level example-bank / held-out split, no leakage |
| 5. Prompting, local inference, parsing | 3.5 – 3.6 | Zero-shot, TF-IDF top-6 few-shot, unique-text cache, 3-stage parser |
| 6. Metrics & cluster bootstrap | 3.3, 3.11 | Weighted / macro F1, activity-instance bootstrap (10 000 resamples), repeatability |
| 7. Privacy | 3.7, 3.8.1 | Egress monitor, tcpdump recipe, cloud counterfactual (calls, bytes, USD) |
| 8. Energy & memory | 3.8, 4.5 | 2 Hz power telemetry, trapezoidal integration, J/inference, tok/s, 70B check |
| 9. Assistive automation | 3.9 | Episodes, 3 disability profiles, cool-downs, 12 scripted scenarios, error propagation |
| 10. Behavioral-change detection | 3.10 | Daily features, robust z-score (median/MAD), perturbation stress tests |
| 11. Experiment runner | 4.1 – 4.3 | Runs the full grid and builds Tables 1–5 |
| 12. Figures | Figs 2–7 | Plots |

## How to run
1. **Quick smoke test (default)**: `DEMO_MODE = True` generates a small synthetic dataset and uses a mock LLM, so the whole notebook runs in minutes with no downloads and no GPU. The numbers it produces are **not** results; they only prove the code runs.
2. **Real experiment**: set `DEMO_MODE = False` (or `ADL_DEMO=0` in the environment), then
   - install [Ollama](https://ollama.com) and pull the models: `ollama pull qwen2.5:7b-instruct-fp16`, `ollama pull mistral:7b-instruct-v0.3-fp16`, `ollama pull llama3.1:8b-instruct-q4_K_M`
   - put the UCI ADL files in `data/uci_adl/` (the notebook tries to download them) and your converted MARBLE files in `data/marble/` (see section 2.3).

> **Assumptions I had to make** (the manuscript does not specify them, so they are all exposed in `CFG`): the bank/test fraction for UCI, whether the text includes time of day (default off, because the paper says the text is "literal about sensor evidence"), the exact phrase templates, rule thresholds, and the MARBLE file layout. Check them against your own experimental code before reporting numbers.

## 1. Setup & configuration

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib scipy openai requests psutil tqdm tiktoken pynvml

In [ ]:
import os, re, json, math, time, hashlib, random, threading, subprocess, itertools, warnings, shutil, zipfile, urllib.request
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

# ---------------------------------------------------------------- switches
DEMO_MODE = os.environ.get("ADL_DEMO", "1") == "1"   # True: synthetic data + mock LLM.  False: real experiment.
RUN_CLOUD_BASELINE = False   # True: also run the *same pipeline* through a cloud LLM (sends window text off-device!)

CFG = dict(
    DATA_DIR   = "data",
    OUT_DIR    = "outputs",
    SEED       = 42,

    # --- Ollama (OpenAI-compatible endpoint) -----------------------------------------------
    OLLAMA_HOST = "http://localhost:11434",           # native API (energy runs, /api/ps)
    OLLAMA_V1   = "http://localhost:11434/v1",        # OpenAI-compatible endpoint (accuracy runs)
    # NOTE: Ollama's *default* tags are 4-bit. The paper serves Qwen/Mistral at "default precision"
    # and only Llama at 4-bit, so fp16 tags are used for the first two. Verify tags with `ollama list`.
    MODELS = {
        "Qwen2.5-7B":  "qwen2.5:7b-instruct-fp16",
        "Mistral-7B":  "mistral:7b-instruct-v0.3-fp16",
        "Llama3.1-8B": "llama3.1:8b-instruct-q4_K_M",
    },
    BIG_MODEL_TAG = "llama3.1:70b-instruct-q5_K_M",   # 70B / 5-bit upper-bound check (section 8.3)
    MAX_NEW_TOKENS = 24,
    WORKERS = 1,              # >1 speeds up accuracy runs; keep 1 for energy runs

    # --- windowing (section 3.4) -----------------------------------------------------------
    UCI_WIN = 60.0,  UCI_STEP = 12.0,       # 60 s window, 80 % overlap -> 12 s step
    MARBLE_WIN = 16.0, MARBLE_STEP = 3.2,   # 16 s window, 80 % overlap -> 3.2 s step
    MIN_LABEL_FRAC = 0.0,                   # min fraction of the window covered by its label (0 = any overlap)
    INCLUDE_TIME_OF_DAY = False,            # add a coarse part-of-day sentence to window text

    # --- partitioning (section 3.3) --------------------------------------------------------
    BANK_FRAC = {"UCI_A": 1/3, "UCI_B": 1/3, "MARBLE": 1/3},   # MARBLE: ~one third is stated in the paper
    # --- prompting (section 3.6) ------------------------------------------------------------
    K_SHOT = 6,
    UCI_FEWSHOT = False,        # the paper reports few-shot only on MARBLE
    # --- uncertainty (section 3.11) ---------------------------------------------------------
    N_BOOT = 10_000,
    REPEAT_N = 100, REPEAT_RUNS = 3,
    # --- energy (section 3.8) ---------------------------------------------------------------
    ENERGY_N = 200, ENERGY_HZ = 2.0, IDLE_SECONDS = 20,
    # --- assistive layer (section 3.9) --------------------------------------------------------
    EP_MIN_DUR = 60.0, EP_MAX_GAP = 120.0,
    ASSIST_MODEL = "Qwen2.5-7B",
    # --- behavioural change (section 3.10) ---------------------------------------------------
    BEH_DAYS = 6, BEH_THRESHOLD = 3.5, BEH_REPS = 20,
    # --- cloud counterfactual (section 3.8.1) ------------------------------------------------
    CLOUD_NAME = "Cloud LLM (same pipeline)", CLOUD_MODEL = "gpt-3.5-turbo-0125", CLOUD_PRICE_IN = 0.50, CLOUD_PRICE_OUT = 1.50,   # USD / 1M tokens
)
if DEMO_MODE:   # shrink everything so the smoke test is fast
    CFG.update(N_BOOT=300, ENERGY_N=10, IDLE_SECONDS=2, REPEAT_N=20, BEH_REPS=5, BEH_DAYS=4)

DATA_DIR, OUT_DIR = Path(CFG["DATA_DIR"]), Path(CFG["OUT_DIR"])
for d in (DATA_DIR, OUT_DIR, OUT_DIR / "cache", OUT_DIR / "figures", OUT_DIR / "tables"):
    d.mkdir(parents=True, exist_ok=True)

SEED = CFG["SEED"]
random.seed(SEED); np.random.seed(SEED)
print(f"DEMO_MODE={DEMO_MODE} | models: {list(CFG['MODELS'])}")

In [ ]:
# ------------------------------------------------------------------ small helpers
_trapz = getattr(np, "trapezoid", None) or np.trapz     # numpy 2 renamed trapz

def to_sec(ts: pd.Series) -> np.ndarray:
    """datetime64 Series -> float seconds since epoch (independent of the datetime unit)."""
    return ((ts - pd.Timestamp("1970-01-01")) / pd.Timedelta(seconds=1)).to_numpy(dtype=float)

def norm_txt(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower().replace("_", " ")).strip()

def oxford_join(items):
    items = list(items)
    if len(items) <= 1: return "".join(items)
    if len(items) == 2: return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"

def sha(s: str) -> str:
    return hashlib.sha1(s.encode()).hexdigest()[:16]

def daypart(sec: float) -> str:
    h = pd.Timestamp(sec, unit="s").hour
    return ("night" if h < 5 else "early morning" if h < 8 else "morning" if h < 12 else
            "afternoon" if h < 17 else "evening" if h < 22 else "night")

## 2. Data loading (Section 3.2)
Both datasets are converted into one common representation so the rest of the pipeline is dataset-agnostic:

* `ds.events`: one row per sensor activation with columns `s, e` (epoch seconds), `phrase`, `place`, `sustained`
* `ds.activities`: one row per labelled activity instance with `s, e, activity, inst`

### 2.1 UCI ADL (Ordóñez et al., 2013): two homes, binary ambient sensors

In [ ]:
@dataclass
class Dataset:
    key: str
    events: pd.DataFrame
    activities: pd.DataFrame
    classes: list
    win: float
    step: float
    home_desc: str
    sensor_desc: str

_DT = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")

def _read_tab_table(path, ncols):
    rows = []
    for line in Path(path).read_text(errors="ignore").splitlines():
        parts = [p.strip() for p in re.split(r"\t+|\s{3,}", line) if p.strip()]
        if len(parts) >= ncols and _DT.match(parts[0]) and _DT.match(parts[1]):
            rows.append(parts[:ncols])
    return rows

_UCI_PHRASES = [  # (substring of the sensor name, phrase, sustained?)
    ("shower", "used the shower", True), ("basin", "used the wash basin", True),
    ("cooktop", "used the cooktop", True), ("maindoor", "opened the main door", False),
    ("fridge", "opened the fridge", False), ("freezer", "opened the freezer", False),
    ("pans", "opened the pans cupboard", False), ("cabinet", "opened a cabinet", False),
    ("cupboard", "opened a cupboard", False), ("toilet", "flushed the toilet", False),
    ("seat", "sat on a seat", True), ("bed", "lay on the bed", True),
    ("microwave", "used the microwave", True), ("toaster", "used the toaster", True),
    ("dishwasher", "used the dishwasher", True), ("washing", "used the washing machine", True),
    ("door", "opened a door", False),
]
def uci_phrase(obj, stype):
    o = obj.lower()
    for key, phrase, sus in _UCI_PHRASES:
        if key in o: return phrase, sus
    return f"triggered the {o} {stype.lower()} sensor", False

def download_uci(dest: Path):
    """Best-effort download of the UCI ADL archive. If it fails, place the 4 files manually."""
    url = "https://archive.ics.uci.edu/static/public/271/activities+of+daily+living+adls+recognition+using+binary+sensors.zip"
    dest.mkdir(parents=True, exist_ok=True)
    z = dest / "uci_adl.zip"
    try:
        urllib.request.urlretrieve(url, z)
        with zipfile.ZipFile(z) as zf: zf.extractall(dest)
        print("downloaded + extracted to", dest)
    except Exception as e:
        print("Automatic download failed:", e,
              "\nDownload manually from https://archive.ics.uci.edu/dataset/271 and put "
              "OrdonezA_ADLs.txt, OrdonezA_Sensors.txt, OrdonezB_ADLs.txt, OrdonezB_Sensors.txt in", dest)

def _find(dirpath: Path, name: str) -> Path:
    hits = list(dirpath.rglob(name))
    if not hits: raise FileNotFoundError(f"{name} not found under {dirpath}")
    return hits[0]

def load_uci(home: str, data_dir: Path) -> Dataset:
    adl = pd.DataFrame(_read_tab_table(_find(data_dir, f"Ordonez{home}_ADLs.txt"), 3), columns=["start", "end", "activity"])
    sen = pd.DataFrame(_read_tab_table(_find(data_dir, f"Ordonez{home}_Sensors.txt"), 5),
                       columns=["start", "end", "obj", "stype", "place"])
    out = []
    for df in (adl, sen):
        for c in ("start", "end"): df[c] = pd.to_datetime(df[c], errors="coerce")
        df.dropna(subset=["start", "end"], inplace=True)          # malformed timestamps are excluded (3.4)
        df.drop(df[df.end < df.start].index, inplace=True)
        df["s"], df["e"] = to_sec(df["start"]), to_sec(df["end"])
        out.append(df.sort_values("s").reset_index(drop=True))
    adl, sen = out
    adl["activity"] = adl["activity"].str.replace("_", " ").str.strip()
    adl["inst"] = np.arange(len(adl))
    ph = [uci_phrase(o, t) for o, t in zip(sen.obj, sen.stype)]
    ev = pd.DataFrame({"s": sen.s, "e": sen.e, "phrase": [p[0] for p in ph],
                       "place": sen.place.str.lower(), "sustained": [p[1] for p in ph]})
    return Dataset(
        key=f"UCI_{home}", events=ev, activities=adl[["s", "e", "activity", "inst"]],
        classes=sorted(adl.activity.unique()), win=CFG["UCI_WIN"], step=CFG["UCI_STEP"],
        home_desc="The home is a single-resident apartment with a bedroom, bathroom, kitchen, living room and entrance.",
        sensor_desc="magnetic contacts on doors, cabinets, cupboards and the fridge; motion (PIR) sensors at the shower, "
                    "wash basin and cooktop; pressure sensors on the bed and seats; appliance electricity sensors; a toilet-flush sensor")

### 2.2 Synthetic demo data (only used when `DEMO_MODE = True`)
Writes small files in the *exact* UCI text format (so the real loader is exercised) plus a MARBLE-style CSV pair.

In [ ]:
DEMO_KEYWORDS = {   # used by the data generator AND by the mock LLM
    "Sleeping": ["lay on the bed"], "Toileting": ["flushed the toilet"], "Showering": ["used the shower"],
    "Grooming": ["wash basin"], "Breakfast": ["cooktop", "toaster"], "Lunch": ["microwave"],
    "Dinner": ["pans cupboard"], "Snack": ["opened a cabinet"], "Leaving": ["main door"], "Spare Time/TV": ["sat on a seat"],
    "Phone call": ["dial a number"], "Watching TV": ["television"], "Using PC": ["office chair"],
    "Setting table": ["drawer"], "Clearing table": ["dishwasher"],
}

def make_demo_uci(dest: Path, home: str, days=4, seed=0):
    rng = np.random.default_rng(seed + (0 if home == "A" else 7))
    base = pd.Timestamp("2011-11-28")
    # (activity, start_h, dur_min, [(sensor, type, place, pulse_every_s, sustained)])
    sched = [("Sleeping", 0.0, 450, [("Bed", "Pressure", "Bedroom", 0, True)]),
             ("Toileting", 7.55, 4, [("Toilet Flush", "Flush", "Bathroom", 0, False)]),
             ("Showering", 7.75, 10, [("Shower", "PIR", "Bathroom", 0, True)]),
             ("Grooming", 7.95, 8, [("Basin", "PIR", "Bathroom", 0, True)]),
             ("Breakfast", 8.2, 30, [("Cooktop", "PIR", "Kitchen", 0, True), ("Toaster", "Electric", "Kitchen", 0, True), ("Fridge", "Magnetic", "Kitchen", 90, False)]),
             ("Leaving", 9.0, 180, [("Maindoor", "Magnetic", "Entrance", 0, False)]),
             ("Lunch", 12.3, 25, [("Microwave", "Electric", "Kitchen", 0, True), ("Fridge", "Magnetic", "Kitchen", 120, False)]),
             ("Spare Time/TV", 13.0, 240, [("Seat", "Pressure", "Living", 0, True)]),
             ("Snack", 17.1, 8, [("Cabinet", "Magnetic", "Kitchen", 45, False)]),
             ("Dinner", 18.5, 40, [("Pans Cupboard", "Magnetic", "Kitchen", 60, False), ("Fridge", "Magnetic", "Kitchen", 150, False)]),
             ("Spare Time/TV", 19.5, 180, [("Seat", "Pressure", "Living", 0, True)]),
             ("Sleeping", 23.0, 59, [("Bed", "Pressure", "Bedroom", 0, True)])]
    if home == "A": sched = [x for x in sched if x[0] != "Dinner"]
    adl_lines, sen_lines = [], []
    f = lambda t: t.strftime("%Y-%m-%d %H:%M:%S")
    for d in range(days):
        for act, h0, dur, sens in sched:
            st = base + pd.Timedelta(days=d, hours=h0) + pd.Timedelta(seconds=int(rng.integers(0, 240)))
            en = st + pd.Timedelta(minutes=dur) - pd.Timedelta(seconds=int(rng.integers(0, 60)))
            adl_lines.append(f"{f(st)}\t{f(en)}\t{act.replace(' ', '_')}")
            for obj, typ, place, every, sus in sens:
                if sus: sen_lines.append(f"{f(st)}\t{f(en)}\t{obj}\t{typ}\t{place}")
                elif act == "Leaving":
                    sen_lines.append(f"{f(st)}\t{f(st + pd.Timedelta(seconds=8))}\t{obj}\t{typ}\t{place}")
                else:
                    t = st + pd.Timedelta(seconds=int(rng.integers(5, 40)))
                    while t < en:
                        sen_lines.append(f"{f(t)}\t{f(t + pd.Timedelta(seconds=int(rng.integers(2, 6))))}\t{obj}\t{typ}\t{place}")
                        t += pd.Timedelta(seconds=every if every else dur * 60)
    dest.mkdir(parents=True, exist_ok=True)
    hdr = "Start time\t\t\tEnd time\t\t\tActivity\n-----------------\t-----------------\t--------\n"
    (dest / f"Ordonez{home}_ADLs.txt").write_text(hdr + "\n".join(adl_lines))
    hdr2 = "Start time\t\t\tEnd time\t\t\tLocation\tType\tPlace\n-----------------\t-----------------\t--------\t----\t-----\n"
    (dest / f"Ordonez{home}_Sensors.txt").write_text(hdr2 + "\n".join(sen_lines))

def make_demo_marble(dest: Path, n_rounds=14, seed=1):
    rng = np.random.default_rng(seed)
    t0 = pd.Timestamp("2022-03-01 09:00:00")
    acts = {"Phone call": [("smartphone_dial", 4)], "Watching TV": [("tv_plug", 0), ("living_couch", 0)],
            "Using PC": [("office_chair", 0)], "Setting table": [("drawer", 25), ("pantry", 40), ("dining_chair", 0)],
            "Clearing table": [("fridge", 30), ("dining_chair", 0), ("dishwasher", 35)]}
    ev, an, t = [], [], t0
    for _ in range(n_rounds):
        for act, sens in acts.items():
            dur = int(rng.integers(50, 110)); st, en = t, t + pd.Timedelta(seconds=dur)
            an.append((st, en, act, "subject1"))
            for name, every in sens:
                if every == 0: ev.append((name, st, en))
                else:
                    x = st + pd.Timedelta(seconds=int(rng.integers(2, 10)))
                    while x < en: ev.append((name, x, x)); x += pd.Timedelta(seconds=every)
            t = en + pd.Timedelta(seconds=int(rng.integers(20, 60)))
    dest.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(ev, columns=["sensor", "start", "end"]).to_csv(dest / "marble_events.csv", index=False)
    pd.DataFrame(an, columns=["start", "end", "activity", "subject"]).to_csv(dest / "marble_activities.csv", index=False)

### 2.3 MARBLE (Arrotta et al., 2022)
MARBLE is distributed in its own raw layout, which I could not verify from here. `load_marble` therefore reads a simple, explicit **two-CSV format** that you produce from the raw release with a few lines of pandas:

* `marble_events.csv`: `sensor,start,end`. Instantaneous events (door open, drawer, smartphone) have `end == start`; sustained states (pressure mat occupied, plug on) have `end > start`.
* `marble_activities.csv`: `start,end,activity,subject`

Edit `MARBLE_PHRASES` so that each of your sensor ids maps to a literal phrase. As in the paper, the two phone labels are merged into "Phone call", "Washing dishes" is excluded, and only single-subject instances are kept.

In [ ]:
MARBLE_PHRASES = {   # sensor id -> (phrase, place, sustained)
    "pantry": ("opened the pantry", "kitchen", False), "drawer": ("opened a drawer", "kitchen", False),
    "medicine_cabinet": ("opened the medicine cabinet", "bathroom", False), "fridge": ("opened the refrigerator", "kitchen", False),
    "stove_plug": ("switched on the stove", "kitchen", True), "tv_plug": ("switched on the television", "living room", True),
    "dining_chair": ("sat on a dining-room chair", "dining room", True), "living_couch": ("sat on the living-room couch", "living room", True),
    "office_chair": ("sat on the office chair", "office", True), "dishwasher": ("opened the dishwasher", "kitchen", False),
    "smartphone_dial": ("used the smartphone to dial a number", "", False),
    "smartphone_ring": ("received an incoming call on the smartphone", "", False),
}
MARBLE_LABEL_MAP = {"phone call": "Phone call", "answering phone": "Phone call", "making phone call": "Phone call"}
MARBLE_EXCLUDE = {"washing dishes"}
MARBLE_SINGLE_SUBJECT_VALUES = None   # e.g. {"subject1"}; None = keep all rows of the activities file

def load_marble(data_dir: Path) -> Dataset:
    ev = pd.read_csv(_find(data_dir, "marble_events.csv"))
    an = pd.read_csv(_find(data_dir, "marble_activities.csv"))
    if MARBLE_SINGLE_SUBJECT_VALUES is not None and "subject" in an:   # keep single-subject instances only
        an = an[an["subject"].isin(MARBLE_SINGLE_SUBJECT_VALUES)]
    for df in (ev, an):
        for c in ("start", "end"): df[c] = pd.to_datetime(df[c], errors="coerce")
        df.dropna(subset=["start", "end"], inplace=True)
        df["s"], df["e"] = to_sec(df["start"]), to_sec(df["end"])
    an["activity"] = an["activity"].astype(str).str.strip()
    an = an[~an.activity.str.lower().isin(MARBLE_EXCLUDE)]
    an["activity"] = [MARBLE_LABEL_MAP.get(a.lower(), a) for a in an.activity]
    an = an.sort_values("s").reset_index(drop=True); an["inst"] = np.arange(len(an))
    ev = ev[ev.sensor.isin(MARBLE_PHRASES)].copy()
    ev["phrase"] = ev.sensor.map(lambda k: MARBLE_PHRASES[k][0]); ev["place"] = ev.sensor.map(lambda k: MARBLE_PHRASES[k][1])
    ev["sustained"] = ev.sensor.map(lambda k: MARBLE_PHRASES[k][2])
    return Dataset(
        key="MARBLE", events=ev.sort_values("s").reset_index(drop=True)[["s", "e", "phrase", "place", "sustained"]],
        activities=an[["s", "e", "activity", "inst"]], classes=sorted(an.activity.unique()),
        win=CFG["MARBLE_WIN"], step=CFG["MARBLE_STEP"],
        home_desc="The home is an instrumented apartment with a kitchen, dining room, living room and office.",
        sensor_desc="magnetic contacts on the pantry, drawers, medicine cabinet and refrigerator; smart plugs on the stove and "
                    "television; pressure mats on dining-room, living-room and office furniture; and a smartphone event channel")

def get_dataset(key: str) -> Dataset:
    if DEMO_MODE:
        d = DATA_DIR / "demo"
        if key.startswith("UCI"):
            h = key[-1]
            if not (d / f"Ordonez{h}_ADLs.txt").exists(): make_demo_uci(d, h)
            return load_uci(h, d)
        if not (d / "marble_events.csv").exists(): make_demo_marble(d)
        return load_marble(d)
    if key.startswith("UCI"):
        d = DATA_DIR / "uci_adl"
        if not list(d.rglob("Ordonez*_ADLs.txt")): download_uci(d)
        return load_uci(key[-1], d)
    return load_marble(DATA_DIR / "marble")

## 3. Windowing and textualization (Section 3.4)
* Sliding windows (60 s / 12 s step for UCI; 16 s / 3.2 s step for MARBLE, both 80 % overlap).
* Label = activity with **maximum temporal overlap**; ties go to the **most recent** activity (prespecified rule).
* Unlabelled windows are excluded; sensor-empty windows are kept and described as "no sensor activity".
* Textualization is deterministic, literal, and never names an activity; a leak check verifies this.

In [ ]:
def build_windows(ds: Dataset, include_tod=False, min_label_frac=0.0) -> pd.DataFrame:
    W, S = ds.win, ds.step
    act, ev = ds.activities, ds.events
    t0 = min(act.s.min(), ev.s.min()); t1 = max(act.e.max(), ev.e.max())
    n = int(np.floor((t1 - t0 - W) / S)) + 1
    ws = t0 + np.arange(n) * S

    # ---- label of every window: maximum temporal overlap, ties -> most recent activity
    best_ov = np.zeros(n); best_start = np.full(n, -np.inf); best_inst = np.full(n, -1, dtype=int)
    for s, e, inst in zip(act.s.to_numpy(), act.e.to_numpy(), act.inst.to_numpy()):
        lo = max(0, int(np.floor((s - t0 - W) / S)) + 1); hi = min(n - 1, int(np.ceil((e - t0) / S)) - 1)
        if hi < lo: continue
        idx = np.arange(lo, hi + 1)
        ov = np.minimum(e, ws[idx] + W) - np.maximum(s, ws[idx])
        better = (ov > best_ov[idx]) | ((ov == best_ov[idx]) & (ov > 0) & (s > best_start[idx]))
        sel = idx[better]
        best_ov[sel] = ov[better]; best_start[sel] = s; best_inst[sel] = inst
    keep = (best_inst >= 0) & (best_ov >= min_label_frac * W) & (best_ov > 0)

    # ---- sensor events falling into each kept window
    win_ev = defaultdict(list)
    for k, (s, e, ph, pl, sus) in enumerate(zip(ev.s, ev.e, ev.phrase, ev.place, ev.sustained)):
        e_eff = max(e, s + 1e-3)
        lo = max(0, int(np.floor((s - t0 - W) / S)) + 1); hi = min(n - 1, int(np.ceil((e_eff - t0) / S)) - 1)
        for i in range(lo, hi + 1):
            if keep[i]: win_ev[i].append((s, e, ph, pl, sus))

    lab = act.set_index("inst").activity
    rows = []
    for i in np.flatnonzero(keep):
        rows.append((ws[i], ws[i] + W, lab[best_inst[i]], int(best_inst[i]), textualize_window(ws[i], W, win_ev.get(i, []), include_tod)))
    return pd.DataFrame(rows, columns=["ws", "we", "label", "inst", "text"])

def textualize_window(ws, W, evs, include_tod=False) -> str:
    head = f"It was {daypart(ws)}. " if include_tod else ""
    if not evs:
        return f"{head}During this {W:g}-second period, no sensor activity was detected."
    evs = sorted(evs, key=lambda x: x[0])
    items, order = Counter(), []
    for s, e, ph, pl, sus in evs:
        if sus:                                   # sustained state (bed, seat, plug, PIR): describe how long
            ov = max(0.0, min(e, ws + W) - max(s, ws))
            ph = f"{ph} {'throughout the period' if ov / W >= 0.9 else 'for part of the period'}"
        key = ph + (f" ({pl})" if pl else "")
        if key not in items: order.append(key)
        items[key] += 1
    parts = [k if items[k] == 1 else f"{k}, {items[k]} times" for k in order]
    return f"{head}During this {W:g}-second period, the resident " + oxford_join(parts) + "."

def label_leak_count(df: pd.DataFrame) -> int:
    """number of distinct (text,label) pairs where the text contains its own activity label"""
    bad = 0
    for text, label in df[["text", "label"]].drop_duplicates().itertuples(index=False):
        if norm_txt(label) and norm_txt(label) in norm_txt(text): bad += 1
    return bad

## 4. Data partitioning and unit of analysis (Section 3.3)
Splits are made at **activity-instance** level (stratified by class), so all overlapping windows of one instance land in the same partition. The *bank* is the only source of few-shot exemplars; the *test* set is the only thing evaluated, so a test window can never be its own exemplar.

In [ ]:
def split_instances(win: pd.DataFrame, bank_frac: float, seed: int) -> pd.DataFrame:
    inst = win.drop_duplicates("inst")[["inst", "label"]]
    rng = np.random.default_rng(seed); bank = set()
    for lab, g in inst.groupby("label"):
        ids = g.inst.to_numpy().copy(); rng.shuffle(ids)
        k = max(1, int(round(len(ids) * bank_frac)))
        if len(ids) > 1: k = min(k, len(ids) - 1)      # keep at least one test instance when possible
        bank.update(ids[:k].tolist())
    win = win.copy(); win["part"] = np.where(win.inst.isin(bank), "bank", "test")
    assert not (set(win[win.part == "bank"].inst) & set(win[win.part == "test"].inst)), "instance leakage between partitions"
    return win

@dataclass
class Prepared:
    ds: Dataset
    win: pd.DataFrame          # all labelled windows with 'part'
    classes: list              # classes present in the test partition (used for metrics); the prompt uses ds.classes
    test: pd.DataFrame
    bank: pd.DataFrame         # unique (text,label) exemplar bank

_prep_cache = {}
def prepare(key: str) -> Prepared:
    if key in _prep_cache: return _prep_cache[key]
    ds = get_dataset(key)
    win = build_windows(ds, CFG["INCLUDE_TIME_OF_DAY"], CFG["MIN_LABEL_FRAC"])
    assert label_leak_count(win) == 0, "label leakage detected in rendered text"
    win = split_instances(win, CFG["BANK_FRAC"][key], SEED)
    test = win[win.part == "test"].reset_index(drop=True)
    bank = win[win.part == "bank"][["text", "label"]].drop_duplicates().reset_index(drop=True)
    p = Prepared(ds, win, sorted(test.label.unique()), test, bank)
    _prep_cache[key] = p
    print(f"[{key}] windows={len(win):,} unique texts={win.text.nunique():,} | test windows={len(test):,} "
          f"(unique texts {test.text.nunique():,}) | bank exemplars={len(bank):,} | classes={len(p.classes)} | leaks=0")
    return p

## 5. Prompting, local inference and output parsing (Sections 3.5 – 3.6)
* Zero-shot: system prompt with home description, sensors and the closed set of labels; answer must be one label in `{braces}`.
* Few-shot: the **K = 6** most similar *bank* exemplars by **TF-IDF cosine similarity** (vectorizer fitted on the bank only, fixed settings), shown before the query, least similar first.
* Greedy decoding, `temperature=0`, fixed seed, through Ollama's OpenAI-compatible endpoint.
* Predictions are cached **by unique text**, then mapped back to every window.
* Parser precedence: **brace -> constrained substring -> stem**. Anything else is counted as an error (never hand-corrected).

In [ ]:
def system_prompt(ds: Dataset, classes) -> str:
    return (f"You are an expert in recognising Activities of Daily Living (ADLs) from smart-home sensor data. {ds.home_desc} "
            f"Available sensors: {ds.sensor_desc}. You will receive a description of the sensor events in one short time window. "
            f"Decide which single activity the resident was performing.\n"
            f"Candidate activities: {' | '.join(classes)}\n"
            f"Answer with exactly one activity from the list, written inside curly braces, for example {{{classes[0]}}}. "
            f"Do not add any explanation.")

def zero_shot_user(text): return f"Sensor description: {text}\nActivity:"

def few_shot_user(text, exemplars):
    blocks = "\n\n".join(f"Sensor description: {t}\nActivity: {{{l}}}" for t, l in exemplars)
    return f"Here are labelled examples from this home:\n\n{blocks}\n\nNow classify the following window.\nSensor description: {text}\nActivity:"

class ExemplarRetriever:
    """TF-IDF cosine retrieval over the (non-overlapping) example bank. Settings are fixed before evaluation."""
    def __init__(self, bank: pd.DataFrame, k: int):
        self.bank, self.k = bank, k
        self.vec = TfidfVectorizer(lowercase=True, stop_words=None, ngram_range=(1, 2), sublinear_tf=True, norm="l2")
        self.Xb = self.vec.fit_transform(bank.text)
    def retrieve(self, texts):
        out = []
        for i in range(0, len(texts), 1000):
            sims = (self.vec.transform(texts[i:i + 1000]) @ self.Xb.T).toarray()
            for row in sims:
                top = np.argsort(-row, kind="stable")[: self.k]          # stable sort => deterministic tie handling
                out.append([(self.bank.text[j], self.bank.label[j]) for j in top[::-1]])   # most similar last
        return out

# ---------------------------------------------------------------------------- LLM back-ends
class OllamaChat:
    def __init__(self, tag, base_url=None, seed=SEED, max_tokens=None, api_key="ollama"):
        from openai import OpenAI
        self.tag, self.seed = tag, seed
        self.max_tokens = max_tokens or CFG["MAX_NEW_TOKENS"]
        self.client = OpenAI(base_url=base_url or CFG["OLLAMA_V1"], api_key=api_key, timeout=180)
    def __call__(self, system, user):
        last = None
        for attempt in range(4):
            try:
                r = self.client.chat.completions.create(
                    model=self.tag, temperature=0, seed=self.seed, max_tokens=self.max_tokens,
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
                return r.choices[0].message.content or ""
            except Exception as e:
                last = e; time.sleep(2 ** attempt)
        raise RuntimeError(f"LLM call failed: {last}")

class MockLLM:
    """Deterministic keyword 'model' used ONLY in DEMO_MODE so the notebook runs without Ollama.
    flip_rate injects reproducible errors so the metrics are not trivially perfect."""
    def __init__(self, name, flip_rate=0.12): self.name, self.flip = name, flip_rate
    def __call__(self, system, user):
        classes = re.search(r"Candidate activities: (.*)\n", system).group(1).split(" | ")
        query = user.rsplit("Sensor description:", 1)[1].split("Activity:")[0].lower()
        scores = {c: sum(k in query for k in DEMO_KEYWORDS.get(c, [])) for c in classes}
        best = max(scores, key=scores.get) if max(scores.values()) > 0 else classes[0]
        h = int(sha(self.name + query), 16)
        if (h % 1000) / 1000 < self.flip: best = classes[h % len(classes)]
        return "{" + best + "}"

def make_backend(model_name: str):
    if model_name == CFG["CLOUD_NAME"] and not DEMO_MODE:      # sends window text to a third party: opt-in only
        return OllamaChat(CFG["CLOUD_MODEL"], base_url="https://api.openai.com/v1", api_key=os.environ["OPENAI_API_KEY"])
    if DEMO_MODE: return MockLLM(model_name, flip_rate={"Qwen2.5-7B": 0.08, "Mistral-7B": 0.14, "Llama3.1-8B": 0.11}.get(model_name, 0.12))
    return OllamaChat(CFG["MODELS"][model_name])

# ---------------------------------------------------------------------------- parser
def parse_output(out: str, classes):
    """returns (label or None, method).  method in {brace, substring, stem, ambiguous, invalid}"""
    ncls = {norm_txt(c): c for c in classes}
    braces = {ncls[norm_txt(b)] for b in re.findall(r"\{([^{}]*)\}", out) if norm_txt(b) in ncls}
    if len(braces) == 1: return next(iter(braces)), "brace"
    if len(braces) > 1: return None, "ambiguous"
    n = norm_txt(out)
    hits = [c for nc, c in ncls.items() if re.search(rf"\b{re.escape(nc)}\b", n)]
    if len(hits) == 1: return hits[0], "substring"
    if len(hits) > 1: return None, "ambiguous"
    stem_hits = [c for nc, c in ncls.items() if re.search(rf"\b{re.escape(nc.split()[0][:5])}", n)]
    if len(set(stem_hits)) == 1: return stem_hits[0], "stem"
    return None, ("ambiguous" if stem_hits else "invalid")

INVALID = "__INVALID__"

# ---------------------------------------------------------------------------- inference with unique-text cache
def _cache_file(key, model, mode): return OUT_DIR / "cache" / f"{key}__{model}__{mode}.json"

def infer_texts(prep: Prepared, model: str, mode: str, texts: list, backend=None, workers=None) -> dict:
    """Raw LLM outputs for the given unique texts (cached on disk; resumable)."""
    assert mode in ("zero", "few")
    backend = backend or make_backend(model)
    cf = _cache_file(prep.ds.key, model, mode)
    cache = json.loads(cf.read_text()) if cf.exists() else {}
    todo = [t for t in dict.fromkeys(texts) if t not in cache]
    if todo:
        sysmsg = system_prompt(prep.ds, prep.ds.classes)
        if mode == "few":
            retr = ExemplarRetriever(prep.bank, CFG["K_SHOT"])
            users = [few_shot_user(t, ex) for t, ex in zip(todo, retr.retrieve(todo))]
        else:
            users = [zero_shot_user(t) for t in todo]
        def job(u): return backend(sysmsg, u)
        w = workers or (1 if DEMO_MODE else CFG["WORKERS"])
        with ThreadPoolExecutor(w) as ex:
            for i, (t, o) in enumerate(zip(todo, tqdm(ex.map(job, users), total=len(todo), desc=f"{prep.ds.key}/{model}/{mode}"))):
                cache[t] = o
                if i % 500 == 499: cf.write_text(json.dumps(cache))
        cf.write_text(json.dumps(cache))
    return cache

def predict_windows(prep: Prepared, model: str, mode: str, frame: pd.DataFrame = None, backend=None):
    """Predicted label for every window of `frame` (default: test windows) + parser statistics."""
    frame = prep.test if frame is None else frame
    cache = infer_texts(prep, model, mode, frame.text.tolist(), backend)
    needed = set(frame.text)
    parsed = {t: parse_output(o, prep.ds.classes) for t, o in cache.items() if t in needed}
    preds = frame.text.map(lambda t: parsed[t][0] or INVALID)
    stats = Counter(m for _, m in parsed.values())                    # per unique text
    return preds, dict(stats)

## 6. Metrics, cluster bootstrap and repeatability (Sections 3.3, 3.11)
* **Weighted F1** (headline, window level) and **macro F1** over the classes present in the test set. Invalid / ambiguous outputs count as errors.
* **95 % CIs** from a **cluster bootstrap at the activity-instance level** (all windows of an instance stay together), 10 000 resamples. For speed, per-instance confusion matrices are pre-computed so a resample is just a weighted sum.
* **Repeatability**: 100 inputs are re-run several times and the parsed label is compared with the first run.

In [ ]:
def f1_scores(y_true, y_pred, classes):
    return dict(
        wf1=f1_score(y_true, y_pred, labels=classes, average="weighted", zero_division=0),
        mf1=f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0),
        per_class=dict(zip(classes, f1_score(y_true, y_pred, labels=classes, average=None, zero_division=0))))

def instance_confusions(test: pd.DataFrame, preds: pd.Series, classes):
    """tensor T[instance, true_class, predicted_class(+1 'invalid' column)]"""
    C = len(classes); cidx = {c: i for i, c in enumerate(classes)}
    insts = test.inst.unique(); pos = {i: k for k, i in enumerate(insts)}
    T = np.zeros((len(insts), C, C + 1))
    yi = test.label.map(cidx).to_numpy()
    pi = preds.map(cidx).fillna(C).astype(int).to_numpy()
    np.add.at(T, (test.inst.map(pos).to_numpy(), yi, pi), 1)
    return T

def wf1_from_conf(conf):
    C = conf.shape[-2]
    tp = np.diagonal(conf[..., :C], axis1=-2, axis2=-1)
    support, pred = conf.sum(-1), conf[..., :C].sum(-2)
    den = support + pred
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    w = support / np.maximum(support.sum(-1, keepdims=True), 1)
    return (w * f1).sum(-1)

def cluster_bootstrap_ci(T, n_boot=None, seed=SEED, alpha=0.05, chunk=500):
    n_boot = n_boot or CFG["N_BOOT"]; rng = np.random.default_rng(seed)
    n = T.shape[0]; flat = T.reshape(n, -1); vals = []
    for b0 in range(0, n_boot, chunk):
        m = min(chunk, n_boot - b0)
        idx = rng.integers(0, n, size=(m, n))
        Wm = np.stack([np.bincount(r, minlength=n) for r in idx]).astype(float)
        vals.append(wf1_from_conf((Wm @ flat).reshape(m, T.shape[1], T.shape[2])))
    vals = np.concatenate(vals)
    return float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))

def repeatability_check(prep: Prepared, model: str, mode: str, n=None, runs=None):
    """Re-run n sampled unique texts `runs` more times (bypassing the cache) and compare the parsed label with run 0."""
    n, runs = n or CFG["REPEAT_N"], runs or CFG["REPEAT_RUNS"]
    rng = np.random.default_rng(SEED)
    texts = list(rng.choice(prep.test.text.unique(), size=min(n, prep.test.text.nunique()), replace=False))
    backend = make_backend(model); sysmsg = system_prompt(prep.ds, prep.ds.classes)
    retr = ExemplarRetriever(prep.bank, CFG["K_SHOT"]) if mode == "few" else None
    exs = retr.retrieve(texts) if retr else [None] * len(texts)
    users = [few_shot_user(t, e) if retr else zero_shot_user(t) for t, e in zip(texts, exs)]
    first = [parse_output(backend(sysmsg, u), prep.ds.classes)[0] for u in users]
    same = np.ones(len(texts), bool)
    for _ in range(runs):
        again = [parse_output(backend(sysmsg, u), prep.ds.classes)[0] for u in users]
        same &= np.array([a == b for a, b in zip(first, again)])
    return dict(n_inputs=len(texts), repeated_runs=runs, identical_label_rate=float(same.mean()))

### 6.1 Self-tests (fast sanity checks of the building blocks)

In [ ]:
_cls = ["Sleeping", "Spare Time/TV", "Phone call"]
assert parse_output("{Sleeping}", _cls) == ("Sleeping", "brace")
assert parse_output("The activity is: Phone call.", _cls) == ("Phone call", "substring")
assert parse_output("sleep", _cls) == ("Sleeping", "stem")
assert parse_output("{Sleeping} {Phone call}", _cls)[0] is None            # two labels -> ambiguous -> counted as error
assert parse_output("no idea", _cls) == (None, "invalid")

_rng = np.random.default_rng(0)                                           # fast confusion-based F1 == sklearn weighted F1
_y = _rng.integers(0, 4, 500); _p = np.where(_rng.random(500) < 0.7, _y, _rng.integers(0, 5, 500))   # label 4 = invalid
_T = np.zeros((1, 4, 5)); np.add.at(_T, (0, _y, _p), 1)
assert abs(wf1_from_conf(_T[0]) - f1_score(_y, _p, labels=[0, 1, 2, 3], average="weighted", zero_division=0)) < 1e-9

print("self-tests passed")

## 7. Privacy measurement and cloud counterfactual (Sections 3.7, 3.8.1, 4.4)
### 7.1 Egress monitoring
Two complementary measurements during a benchmark run:
1. **`EgressMonitor`** (no root needed): polls every 0.5 s all sockets of the Ollama processes and of this Python kernel and records any connection whose remote address is not loopback / private, plus byte-counter deltas on non-loopback interfaces.
2. **Packet capture** (needs `sudo`; recipe in the next cell) on `lo` at the inference port and on the external interface, exactly as in Section 3.7.

Model *downloads* (`ollama pull`) legitimately use the network, so pull everything **before** starting the monitor.

In [ ]:
import ipaddress, psutil

def _is_local(ip: str) -> bool:
    try: a = ipaddress.ip_address(ip)
    except ValueError: return True
    return a.is_loopback or a.is_private or a.is_link_local

class EgressMonitor(threading.Thread):
    def __init__(self, interval=0.5, procs=("ollama",), port=11434):
        super().__init__(daemon=True)
        self.interval, self.procs, self.port = interval, procs, port
        self.stop_evt = threading.Event(); self.flagged = set(); self.local_conns = 0; self.samples = 0
        self.nic0 = self._nic(); self.nic1 = None
    @staticmethod
    def _nic():
        return {k: (v.bytes_sent, v.bytes_recv) for k, v in psutil.net_io_counters(pernic=True).items() if not k.startswith("lo")}
    def _pids(self):
        pids = {os.getpid()}
        for p in psutil.process_iter(["name"]):
            if any(n in (p.info["name"] or "").lower() for n in self.procs): pids.add(p.pid)
        return pids
    def run(self):
        while not self.stop_evt.is_set():
            try:
                pids = self._pids()
                for c in psutil.net_connections(kind="inet"):
                    if c.pid in pids and c.raddr:
                        if _is_local(c.raddr.ip): self.local_conns += 1
                        else: self.flagged.add((c.pid, c.raddr.ip, c.raddr.port, c.status))
                self.samples += 1
            except (psutil.AccessDenied, PermissionError): pass     # on macOS run the kernel with sudo, or rely on pcap
            time.sleep(self.interval)
    def __enter__(self): self.start(); return self
    def __exit__(self, *a):
        self.stop_evt.set(); self.join(2); self.nic1 = self._nic()
    def report(self):
        nic = {k: dict(sent=self.nic1[k][0] - self.nic0[k][0], recv=self.nic1[k][1] - self.nic0[k][1]) for k in self.nic1 if k in self.nic0}
        return dict(monitor_samples=self.samples, loopback_or_private_conns_seen=self.local_conns,
                    external_connections=sorted(self.flagged), n_external_connections=len(self.flagged),
                    non_loopback_nic_bytes_during_run=nic,
                    note="NIC byte counters include unrelated OS traffic (DNS/NTP/updates); use pcap to attribute it.")

PCAP_RECIPE = """
# --- run these in a terminal with sudo, BEFORE starting the benchmark cell, stop them afterwards -------------------
sudo tcpdump -i lo   -nn -w lo_inference.pcap   'tcp port 11434'          # app <-> model server traffic (local only)
sudo tcpdump -i any  -nn -w external.pcap       'not (host 127.0.0.1 or net 10.0.0.0/8 or net 172.16.0.0/12 or net 192.168.0.0/16)'
# --- afterwards ---------------------------------------------------------------------------------------------------
tcpdump -nn -r external.pcap | head          # expect only DNS / NTP / OS update chatter, no payload to a model API
tcpdump -nn -r external.pcap 'tcp' | wc -l   # TCP packets leaving the box; inspect their destinations
"""
print(PCAP_RECIPE)

### 7.2 Cloud counterfactual (Section 3.8.1)
Calls per day follow directly from the windowing cadence (86 400 s / 3.2 s = **27 000** at the MARBLE step). Bytes use the real rendered texts; tokens use `tiktoken` (`cl100k_base`) on the *actual prompts* (system prompt + exemplars + query); cost multiplies by the published per-token price. Both the "no caching" figure used in the paper and a "cache-aware" figure (only first occurrences of identical texts are sent) are reported.

In [ ]:
def count_tokens(text: str) -> int:
    try:
        import tiktoken
        return len(tiktoken.get_encoding("cl100k_base").encode(text))
    except Exception:
        return max(1, len(text) // 4)          # offline fallback (~4 chars / token)

def cloud_counterfactual(prep: Prepared, mode="zero", avg_out_tokens=6, n_sample=500):
    step = prep.ds.step
    calls_day = 86400 / step
    texts = prep.test.text
    avg_bytes = float(texts.map(lambda t: len(t.encode())).mean())
    sysmsg = system_prompt(prep.ds, prep.ds.classes)
    samp = list(texts.drop_duplicates().sample(min(n_sample, texts.nunique()), random_state=SEED))
    if mode == "few":
        users = [few_shot_user(t, e) for t, e in zip(samp, ExemplarRetriever(prep.bank, CFG["K_SHOT"]).retrieve(samp))]
    else:
        users = [zero_shot_user(t) for t in samp]
    avg_in = float(np.mean([count_tokens(sysmsg + "\n" + u) for u in users]))
    uniq_frac = prep.test.text.nunique() / len(prep.test)          # share of windows whose text is new (cache-aware)
    def usd(calls): return calls * (avg_in * CFG["CLOUD_PRICE_IN"] + avg_out_tokens * CFG["CLOUD_PRICE_OUT"]) / 1e6
    return dict(
        windowing_step_s=step, api_calls_per_day=calls_day, activity_text_bytes_per_call=avg_bytes,
        activity_text_MB_per_day=calls_day * avg_bytes / 1e6, external_disclosures_per_year=calls_day * 365,
        avg_input_tokens=avg_in, avg_output_tokens=avg_out_tokens,
        usd_per_day=usd(calls_day), usd_per_year=usd(calls_day) * 365,
        cache_aware_calls_per_day=calls_day * uniq_frac, cache_aware_usd_per_year=usd(calls_day * uniq_frac) * 365,
        on_device=dict(api_calls_per_day=0, bytes_egress_per_day=0, external_disclosures_per_year=0, usd_per_year=0))

## 8. Energy and memory (Sections 3.8, 4.5)
### 8.1 Power telemetry (2 Hz) and trapezoidal integration
Reads GPU board power through NVML (`pynvml`), falling back to `nvidia-smi`. Per inference the power trace is integrated with the trapezoidal rule between the request's start and end timestamps (end points are interpolated). **Gross** = ∫P dt, **net** = ∫(P − P_idle) dt with P_idle measured before the run. Throughput = Ollama's reported `eval_count / eval_duration`. GPU telemetry excludes CPU / system overhead, so absolute values are a lower bound for whole-system energy.

In [ ]:
import requests

class PowerMonitor:
    def __init__(self, hz=None):
        self.dt = 1.0 / (hz or CFG["ENERGY_HZ"]); self.samples = []; self._stop = threading.Event(); self._thr = None
        self.mode = None
        try:
            import pynvml; pynvml.nvmlInit(); self._h = pynvml.nvmlDeviceGetHandleByIndex(0); self._nv = pynvml; self.mode = "nvml"
            self._nv.nvmlDeviceGetPowerUsage(self._h)
        except Exception:
            self.mode = "nvidia-smi" if shutil.which("nvidia-smi") else "unavailable"
    def read_w(self):
        if self.mode == "nvml": return self._nv.nvmlDeviceGetPowerUsage(self._h) / 1000.0
        if self.mode == "nvidia-smi":
            out = subprocess.run(["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
            return float(out.strip().splitlines()[0])
        return float("nan")
    def _loop(self):
        while not self._stop.is_set():
            t = time.time()
            try: self.samples.append((t, self.read_w()))
            except Exception: pass
            time.sleep(max(0.0, self.dt - (time.time() - t)))
    def __enter__(self): self.samples = []; self._stop.clear(); self._thr = threading.Thread(target=self._loop, daemon=True); self._thr.start(); return self
    def __exit__(self, *a): self._stop.set(); self._thr.join(2)
    def energy_j(self, t_a, t_b, baseline_w=0.0):
        if len(self.samples) < 2 or not np.isfinite(self.samples[0][1]): return float("nan")
        ts, ps = np.array(self.samples).T
        grid = np.concatenate([[t_a], ts[(ts > t_a) & (ts < t_b)], [t_b]])
        p = np.interp(grid, ts, ps) - baseline_w
        return float(_trapz(p, grid))

def ollama_generate(tag, system, user, seed=SEED):
    r = requests.post(f"{CFG['OLLAMA_HOST']}/api/chat", timeout=300, json=dict(
        model=tag, stream=False, messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        options=dict(temperature=0, seed=seed, num_predict=CFG["MAX_NEW_TOKENS"])))
    r.raise_for_status(); return r.json()

def measure_energy(prep: Prepared, model: str, n=None):
    n = n or CFG["ENERGY_N"]
    texts = list(prep.test.text.drop_duplicates().sample(min(n, prep.test.text.nunique()), random_state=SEED))
    sysmsg = system_prompt(prep.ds, prep.ds.classes)
    pm = PowerMonitor()
    if DEMO_MODE:                                   # no GPU / Ollama in the smoke test: emit a clearly-labelled synthetic row
        rng = np.random.default_rng(int(sha(model), 16) % 2**32)
        g = float(rng.uniform(25, 45)); return dict(model=model, gross_J=g, net_J=g - 4.0, tok_s=float(rng.uniform(200, 270)), n=len(texts), telemetry="SYNTHETIC (demo)")
    tag = CFG["MODELS"][model]
    for t in texts[:3]: ollama_generate(tag, sysmsg, zero_shot_user(t))          # warm-up: load weights
    with pm:
        time.sleep(CFG["IDLE_SECONDS"])
    idle_w = float(np.nanmean([p for _, p in pm.samples])) if pm.samples else float("nan")
    gross, net, toks = [], [], []
    with pm:
        time.sleep(1.0 / CFG["ENERGY_HZ"] * 2)
        for t in tqdm(texts, desc=f"energy/{model}"):
            a = time.time(); res = ollama_generate(tag, sysmsg, zero_shot_user(t)); b = time.time()
            gross.append(pm.energy_j(a, b)); net.append(pm.energy_j(a, b, idle_w))
            toks.append(res["eval_count"] / (res["eval_duration"] / 1e9))
    return dict(model=model, gross_J=float(np.nanmean(gross)), net_J=float(np.nanmean(net)), tok_s=float(np.mean(toks)),
                idle_W=idle_w, n=len(texts), telemetry=pm.mode)

### 8.2 Memory-feasibility check (the "70B does not fit" argument)
Weights alone: `params × bits / 8`. 70 B at 5 bit ≈ 43.8 GB (≈ 45–50 GB with quantization metadata); the ≈ 123 GB observed at inference time also includes runtime buffers and the KV cache, so it must not be read as the weight size. `ollama ps` reports the true resident size.

In [ ]:
def weight_gb(params_b: float, bits: float) -> float: return params_b * 1e9 * bits / 8 / 1e9

def memory_report(tags: dict):
    rows = [dict(model="70B @ 5-bit (weights only)", GB=weight_gb(70, 5)), dict(model="7B @ 16-bit (weights only)", GB=weight_gb(7, 16)),
            dict(model="8B @ 4-bit (weights only)", GB=weight_gb(8, 4))]
    if not DEMO_MODE:
        try:
            for m in requests.get(f"{CFG['OLLAMA_HOST']}/api/ps", timeout=5).json().get("models", []):
                rows.append(dict(model=f"resident in Ollama: {m['name']}", GB=m["size"] / 1e9, VRAM_GB=m.get("size_vram", 0) / 1e9))
        except Exception as e: print("ollama /api/ps unavailable:", e)
    return pd.DataFrame(rows)

## 9. Disability-conditioned assistive automation (Section 3.9)
Pipeline: recognised windows → **episodes** (merge consecutive equal labels, combine gaps < 120 s, drop episodes < 60 s) → **transparent rule engine** for three profiles (cognitive, visual, mobility, ≥ 3 rules each) → per-rule **cool-downs** → **audit log**. Safety-critical actions are only ever *flagged for human review*; nothing is sent to a caregiver automatically.

Rule thresholds are investigator-defined engineering choices (a proof-of-concept specification, *not* clinical requirements).

In [ ]:
GROUP_KEYS = [("sleep", ["sleep"]), ("leaving", ["leav", "outing", "depart"]), ("hygiene", ["shower", "groom", "toilet", "bath"]),
              ("snack", ["snack"]), ("meal", ["breakfast", "lunch", "dinner", "meal", "cook", "prepar"]),
              ("relax", ["spare", "tv", "relax", "television", "watch", "pc"])]
def group_of(label: str) -> str:
    toks = norm_txt(label).split()
    for g, keys in GROUP_KEYS:
        if any(t.startswith(k) for t in toks for k in keys): return g
    return "other"

@dataclass
class Episode:
    activity: str; start: float; end: float
    @property
    def group(self): return group_of(self.activity)
    @property
    def dur(self): return self.end - self.start

def build_episodes(win: pd.DataFrame, label_col: str, W: float, S: float, min_dur=None, max_gap=None):
    """Each window 'owns' the step around its centre. Merge equal labels, combine gaps < max_gap, drop runs < min_dur, re-merge."""
    min_dur = CFG["EP_MIN_DUR"] if min_dur is None else min_dur
    max_gap = CFG["EP_MAX_GAP"] if max_gap is None else max_gap
    df = win.sort_values("ws"); runs = []
    for ws, lab in zip(df.ws.to_numpy(), df[label_col].to_numpy()):
        s, e = ws + W / 2 - S / 2, ws + W / 2 + S / 2
        if lab == INVALID: continue
        if runs and runs[-1].activity == lab and s - runs[-1].end <= 0.5 * S: runs[-1].end = e
        else: runs.append(Episode(lab, s, e))
    def merge(rs):
        out = []
        for r in rs:
            if out and out[-1].activity == r.activity and r.start - out[-1].end < max_gap: out[-1].end = r.end
            else: out.append(Episode(r.activity, r.start, r.end))
        return out
    runs = merge(runs); runs = [r for r in runs if r.dur >= min_dur]
    return merge(runs)

def hour_of(sec: float) -> float:
    ts = pd.Timestamp(sec, unit="s"); return ts.hour + ts.minute / 60

@dataclass
class Action:
    t: float; profile: str; rule: str; activity: str; kind: str; message: str; rationale: str; human_review: bool
    auto_sent: bool = False          # invariant: caregiver-facing actions are NEVER auto-sent

@dataclass
class Rule:
    id: str; profile: str; name: str; cooldown_s: float
    check: callable        # (i, episodes) -> None | (t, kind, message, rationale, human_review)

def _nxt(eps, i): return eps[i + 1] if i + 1 < len(eps) else None

# --------------------------------------------------------------------- COGNITIVE profile
def c1_med_reminder(i, eps):
    e = eps[i]
    if e.group == "meal" and "breakfast" in norm_txt(e.activity):
        return e.end, "speak+notify", "Reminder: it is time to take your morning medication.", "Breakfast episode ended; medication is scheduled after breakfast.", False
def c2_unattended_cooking(i, eps, window_s=15 * 60):
    e, n = eps[i], _nxt(eps, i)
    if e.group == "meal" and n and n.start - e.end <= window_s and n.group in ("leaving", "sleep"):
        return n.start, "caregiver_flag", f"Possible unattended cooking: {e.activity} was followed by '{n.activity}' within {window_s // 60} min.", \
               "Meal episode not followed by an eating/resting episode within the configured window.", True
def c3_night_activity(i, eps):
    e = eps[i]
    if 1 <= hour_of(e.start) < 5 and e.group not in ("sleep",) and not ("toilet" in norm_txt(e.activity) and e.dur < 600):
        return e.start, "caregiver_flag", f"Night-time activity recognised ({e.activity}) at {pd.Timestamp(e.start, unit='s'):%H:%M}.", \
               "Non-sleep activity recognised in the early-morning hours.", True

# --------------------------------------------------------------------- VISUAL profile
def v1_announce(i, eps):
    e = eps[i]; return e.start, "speak", f"Now: {e.activity}.", "Spoken narration of the context transition.", False
def v2_cooking_cue(i, eps):
    e = eps[i]
    if e.group == "meal": return e.start, "speak", "Cooking activity detected. Take care: the cooktop and appliances may be hot.", "Meal-preparation episode started.", False
def v3_wake_summary(i, eps):
    e = eps[i]
    if e.group == "sleep": return e.end, "speak", f"You are awake. The time is {pd.Timestamp(e.end, unit='s'):%H:%M}.", "Sleep episode ended; orientation announcement.", False

# --------------------------------------------------------------------- MOBILITY profile
def m1_departure(i, eps):
    e = eps[i]
    if e.group == "leaving": return e.start, "speak+notify", "You are leaving home: take your keys, phone and mobility aid.", "Leaving-home episode started.", False
def m2_night_path(i, eps):
    e = eps[i]
    if e.group == "sleep" and (hour_of(e.end) >= 22 or hour_of(e.end) < 6):
        return e.end, "automation", "Turn ON the bedroom-to-bathroom path lighting.", "Transition out of sleep at night (fall-risk situation).", False
def m3_inactivity(i, eps, limit_s=2 * 3600):
    e = eps[i]
    if e.group == "relax" and e.dur >= limit_s:
        return e.start + limit_s, "caregiver_flag", f"No change of activity for {limit_s // 3600} h ({e.activity}). Suggest a check-in.", \
               "Prolonged sedentary episode.", True

RULES = [Rule("C1", "cognitive", "Medication reminder after breakfast", 6 * 3600, c1_med_reminder),
         Rule("C2", "cognitive", "Unattended-cooking flag", 3600, c2_unattended_cooking),
         Rule("C3", "cognitive", "Night-time activity flag", 3 * 3600, c3_night_activity),
         Rule("V1", "visual", "Spoken context announcement", 60, v1_announce),
         Rule("V2", "visual", "Cooking safety cue", 1800, v2_cooking_cue),
         Rule("V3", "visual", "Wake-up orientation", 6 * 3600, v3_wake_summary),
         Rule("M1", "mobility", "Departure reminder", 600, m1_departure),
         Rule("M2", "mobility", "Night path lighting", 1800, m2_night_path),
         Rule("M3", "mobility", "Prolonged-inactivity check-in", 2 * 3600, m3_inactivity)]
PROFILES = ["cognitive", "visual", "mobility"]

def run_rules(eps, profiles=PROFILES, rules=RULES):
    """Evaluate every rule on every episode, then apply per-(profile, rule) cool-downs in time order. Returns the audit log."""
    cand = []
    for r in rules:
        if r.profile not in profiles: continue
        for i in range(len(eps)):
            res = r.check(i, eps)
            if res: cand.append(Action(res[0], r.profile, r.id, eps[i].activity, res[1], res[2], res[3], res[4]))
    cand.sort(key=lambda a: (a.t, a.rule)); last, out = {}, []
    cd = {r.id: r.cooldown_s for r in rules}
    for a in cand:
        if a.t - last.get(a.rule, -1e18) >= cd[a.rule]: out.append(a); last[a.rule] = a.t
    assert not any(a.auto_sent for a in out)
    return out

def audit_log(actions) -> pd.DataFrame:
    return pd.DataFrame([dict(time=pd.Timestamp(a.t, unit="s"), profile=a.profile, rule=a.rule, activity=a.activity, kind=a.kind,
                              human_review=a.human_review, auto_sent=a.auto_sent, message=a.message, rationale=a.rationale) for a in actions])

### 9.1 Scenario-based rule verification (12 scripted scenarios: 9 positive, 3 negative controls)
Scenarios are fed as *window-level* label streams through the same episode builder and engine, so the min-duration / gap logic is exercised too. This verifies software logic only, not clinical effectiveness.

In [ ]:
def scenario_windows(spec, W=60.0, S=12.0, day="2025-01-06"):
    """spec: list of (activity, 'HH:MM', duration_s). Windows are labelled by the activity covering the window centre."""
    t0 = to_sec(pd.Series([pd.Timestamp(day)]))[0]
    segs = [(a, t0 + pd.Timedelta(f"{h}:00").total_seconds(), d) for a, h, d in spec]
    start, end = min(s for _, s, _ in segs), max(s + d for _, s, d in segs)
    rows = []
    for ws in np.arange(start - W / 2, end - W / 2, S):
        c = ws + W / 2; lab = next((a for a, s, d in segs if s <= c < s + d), None)
        if lab: rows.append((ws, lab))
    return pd.DataFrame(rows, columns=["ws", "pred"])

SCENARIOS = [  # (name, spec, expected rule ids that MUST fire, rule ids that MUST NOT fire)
    ("P1 breakfast -> medication reminder", [("Breakfast", "08:00", 1200)], {"C1"}, set()),
    ("P2 cooking then leaving -> unattended flag", [("Breakfast", "08:00", 1200), ("Leaving", "08:25", 3600)], {"C2"}, set()),
    ("P3 night TV at 02:30 -> night-activity flag", [("Spare Time/TV", "02:30", 1800)], {"C3"}, set()),
    ("P4 shower -> spoken announcement", [("Showering", "07:00", 600)], {"V1"}, set()),
    ("P5 lunch -> cooking cue", [("Lunch", "12:00", 1500)], {"V2"}, set()),
    ("P6 wake in the morning -> orientation", [("Sleeping", "00:00", 7 * 3600)], {"V3"}, set()),
    ("P7 leaving -> departure reminder", [("Leaving", "14:00", 3600)], {"M1"}, set()),
    ("P8 wake at 03:00 -> path lighting", [("Sleeping", "00:00", 3 * 3600), ("Toileting", "03:00", 300)], {"M2"}, set()),
    ("P9 3 h sedentary -> inactivity check-in", [("Spare Time/TV", "14:00", 3 * 3600)], {"M3"}, set()),
    ("N1 breakfast then TV (attended) -> no unattended flag", [("Breakfast", "08:00", 1200), ("Spare Time/TV", "08:25", 2400)], set(), {"C2"}),
    ("N2 afternoon TV 1 h -> no night / inactivity flag", [("Spare Time/TV", "14:00", 3600)], set(), {"C3", "M3"}),
    ("N3 40 s 'Leaving' blip -> filtered, no reminder", [("Spare Time/TV", "14:00", 1800), ("Leaving", "14:30", 40), ("Spare Time/TV", "14:31", 1800)], set(), {"M1"}),
]

def run_scenarios():
    rows = []
    for name, spec, must, mustnot in SCENARIOS:
        fired = {a.rule for a in run_rules(build_episodes(scenario_windows(spec), "pred", 60.0, 12.0))}
        rows.append(dict(scenario=name, type="positive" if must else "negative", expected=sorted(must), forbidden=sorted(mustnot),
                         fired=sorted(fired), passed=(must <= fired) and not (mustnot & fired)))
    df = pd.DataFrame(rows)
    pos, neg = df[df.type == "positive"], df[df.type == "negative"]
    summary = dict(n_positive=len(pos), expected_action_recall=float(pos.passed.mean()),
                   n_negative=len(neg), false_trigger_rate=float(1 - neg.passed.mean()))
    return df, summary

In [ ]:
# self-test: a 40 s blip is filtered out and its neighbours are merged into one episode
_w = scenario_windows([("Spare Time/TV", "14:00", 1800), ("Leaving", "14:30", 40), ("Spare Time/TV", "14:31", 1800)])
assert [e.activity for e in build_episodes(_w, "pred", 60.0, 12.0)] == ["Spare Time/TV"]
print("episode-builder self-test passed")

### 9.2 Recognition-error propagation (Table 4) and controlled robustness tests
Actions produced from the **recognised** stream are matched (same rule, ±5 min) with actions produced from the **ground-truth** stream. Unmatched predicted actions are *error-triggered*; unmatched truth actions are *missed*. The sweep then perturbs the ground truth in a controlled way (label substitution / missed detection) and reports expected-action recall, false-trigger rate and missed-action rate.

In [ ]:
def match_actions(pred, truth, tol=300.0):
    used, matched = set(), 0
    for a in pred:
        for j, b in enumerate(truth):
            if j not in used and a.rule == b.rule and abs(a.t - b.t) <= tol: used.add(j); matched += 1; break
    return dict(total_actions=len(pred), correct_recognition_actions=matched, recognition_error_actions=len(pred) - matched,
                error_triggered_rate=(len(pred) - matched) / max(1, len(pred)), truth_actions=len(truth),
                missed_actions=len(truth) - matched, expected_action_recall=matched / max(1, len(truth)))

def robustness_sweep(win_true: pd.DataFrame, classes, W, S, rates=(0.0, 0.05, 0.1, 0.2, 0.3), seed=SEED):
    base = win_true.assign(pred=win_true.label)
    truth = run_rules(build_episodes(base, "pred", W, S))
    rows = []
    for mode in ("substitution", "missed_detection"):
        for p in rates:
            rng = np.random.default_rng(seed); d = base.copy(); flip = rng.random(len(d)) < p
            if mode == "substitution": d.loc[flip, "pred"] = rng.choice(classes, size=int(flip.sum()))
            else: d.loc[flip, "pred"] = INVALID                        # no detection -> window dropped
            m = match_actions(run_rules(build_episodes(d, "pred", W, S)), truth)
            rows.append(dict(perturbation=mode, rate=p, expected_action_recall=m["expected_action_recall"],
                             false_trigger_rate=m["error_triggered_rate"], missed_action_rate=1 - m["expected_action_recall"]))
    return pd.DataFrame(rows)

## 10. Behavioral-change detection (Section 3.10): proof of concept
Daily features from recognised activities → robust z-score of each day against the resident's **other** days (median / MAD with a per-feature minimum-scale floor) → flag days whose largest |z| exceeds a pre-specified threshold. With no labelled longitudinal changes available, the method is **stress-tested by injecting perturbations** (delayed meals, dropped hygiene, fragmented sleep) into real days. These are simulation-based tests, not validation against clinical labels.

In [ ]:
FEATURES = ["sleep_h", "sedentary_h", "n_hygiene", "n_meals", "n_departures", "first_hygiene_h", "first_meal_h", "transition_entropy"]
FLOORS = dict(sleep_h=0.5, sedentary_h=0.5, n_hygiene=0.5, n_meals=0.5, n_departures=0.5, first_hygiene_h=0.5, first_meal_h=0.5, transition_entropy=0.1)

def episodes_df(eps) -> pd.DataFrame:
    return pd.DataFrame([dict(activity=e.activity, group=e.group, start=e.start, end=e.end) for e in eps])

def daily_features(edf: pd.DataFrame) -> pd.DataFrame:
    edf = edf.copy(); edf["day"] = pd.to_datetime(edf.start, unit="s").dt.normalize()
    rows = {}
    for day, g in edf.sort_values("start").groupby("day"):
        dur = lambda grp: float((g[g.group == grp].end - g[g.group == grp].start).sum() / 3600)
        hyg, meal = g[g.group == "hygiene"], g[g.group == "meal"]
        fh = lambda x: (hour_of(x.start.min()) if len(x) else np.nan)
        trans = Counter(zip(g.group.iloc[:-1], g.group.iloc[1:])); tot = sum(trans.values())
        ent = float(-sum(c / tot * math.log2(c / tot) for c in trans.values())) if tot else 0.0
        rows[day] = dict(sleep_h=dur("sleep"), sedentary_h=dur("relax"), n_hygiene=len(hyg), n_meals=len(meal),
                         n_departures=int((g.group == "leaving").sum()), first_hygiene_h=fh(hyg), first_meal_h=fh(meal), transition_entropy=ent)
    return pd.DataFrame(rows).T.sort_index()

def robust_z(day_row: pd.Series, others: pd.DataFrame) -> pd.Series:
    others = others.fillna(24.0); x = day_row.fillna(24.0)         # "event never happened" = end of day
    med = others.median(); mad = 1.4826 * (others - med).abs().median()
    scale = np.maximum(mad, pd.Series(FLOORS))
    return (x - med) / scale

def flag_days(F: pd.DataFrame, thr=None):
    thr = thr or CFG["BEH_THRESHOLD"]; out = {}
    for d in F.index:
        z = robust_z(F.loc[d], F.drop(index=d)); out[d] = dict(max_abs_z=float(z.abs().max()), top_feature=z.abs().idxmax(), flagged=bool(z.abs().max() >= thr))
    return pd.DataFrame(out).T

# ----------------------------------------------------------------- injected perturbations (applied to one day's episode table)
def p_delay_meals(g, rng):
    g = g.copy(); h = rng.uniform(2.0, 4.0) * 3600
    m = g.group == "meal"; g.loc[m, ["start", "end"]] += h; return g
def p_drop_hygiene(g, rng): return g[g.group != "hygiene"].copy()
def p_fragment_sleep(g, rng):
    rows = []
    for r in g.itertuples(index=False):
        if r.group == "sleep" and r.end - r.start > 3 * 3600:
            k = int(rng.integers(3, 6)); cuts = np.linspace(r.start, r.end, k + 2)
            for a, b in zip(cuts[:-1], cuts[1:]):
                rows.append(dict(activity=r.activity, group="sleep", start=a, end=b - 900))     # 15-min awake gap after each part
                rows.append(dict(activity="Awake", group="other", start=b - 900, end=b))
        else: rows.append(r._asdict())
    return pd.DataFrame(rows)
PERTURBATIONS = {"delayed_meals": p_delay_meals, "dropped_hygiene": p_drop_hygiene, "fragmented_sleep": p_fragment_sleep}

def behaviour_experiment(edf: pd.DataFrame, n_days=None, reps=None, seed=SEED, thr=None):
    """Select the n_days best-covered days, flag them naturally, then inject each perturbation reps times per day."""
    n_days, reps, thr = n_days or CFG["BEH_DAYS"], reps or CFG["BEH_REPS"], thr or CFG["BEH_THRESHOLD"]
    edf = edf.copy(); edf["day"] = pd.to_datetime(edf.start, unit="s").dt.normalize()
    cover = edf.assign(d=edf.end - edf.start).groupby("day").d.sum().sort_values(ascending=False)
    days = sorted(cover.index[:n_days]); sub = edf[edf.day.isin(days)]
    F = daily_features(sub.drop(columns="day")); nat = flag_days(F, thr)
    rng = np.random.default_rng(seed); det = defaultdict(list)
    for d in days:
        g = sub[sub.day == d].drop(columns="day")
        for name, fn in PERTURBATIONS.items():
            for _ in range(reps):
                fp = daily_features(fn(g, rng))
                if d not in fp.index: det[name].append(True); continue      # the whole day vanished: trivially detectable
                Fp = F.copy(); Fp.loc[d] = fp.loc[d]
                z = robust_z(Fp.loc[d], Fp.drop(index=d)); det[name].append(bool(z.abs().max() >= thr))
    return dict(days=[str(d.date()) for d in days], features=F, natural=nat, baseline_flag_rate=float(nat.flagged.mean()),
                detection={k: float(np.mean(v)) for k, v in det.items()})

## 11. Experiment runner (Algorithm 1) and results tables
### 11.1 Recognition grid: datasets × models × prompting (Sections 4.1 – 4.3)

In [ ]:
RESULTS = {}      # (dataset, model, mode) -> dict

def evaluate(key, model, mode):
    prep = prepare(key)
    preds, stats = predict_windows(prep, model, mode)
    sc = f1_scores(prep.test.label, preds, prep.classes)
    T = instance_confusions(prep.test, preds, prep.classes)
    lo, hi = cluster_bootstrap_ci(T)
    RESULTS[(key, model, mode)] = dict(wf1=sc["wf1"], mf1=sc["mf1"], ci=(lo, hi), per_class=sc["per_class"], parse=stats,
                                       n_windows=len(prep.test), n_instances=T.shape[0], preds=preds)
    return RESULTS[(key, model, mode)]

def run_grid():
    models = list(CFG["MODELS"]) + ([CFG["CLOUD_NAME"]] if RUN_CLOUD_BASELINE else [])
    for key in ("UCI_A", "UCI_B"):
        for m in models:
            evaluate(key, m, "zero")
            if CFG["UCI_FEWSHOT"]: evaluate(key, m, "few")
    for m in models:
        for mode in ("zero", "few"): evaluate("MARBLE", m, mode)
run_grid()

In [ ]:
def table1():
    rows = []
    for m in list(CFG["MODELS"]) + ([CFG["CLOUD_NAME"]] if RUN_CLOUD_BASELINE else []):
        g = lambda k, f: RESULTS.get((k, m, "zero"), {}).get(f, np.nan)
        rows.append({"Model": m, "Deployment": "Cloud" if m == CFG["CLOUD_NAME"] else "Local", "Prompting": "Zero-shot",
                     "Home A wF1": g("UCI_A", "wf1"), "Home A mF1": g("UCI_A", "mf1"), "Home B wF1": g("UCI_B", "wf1"), "Home B mF1": g("UCI_B", "mf1")})
    rows.append({"Model": "gpt-3.5-turbo (published, Civitarese et al. 2025)", "Deployment": "Cloud", "Prompting": "Zero-shot",
                 "Home A wF1": 0.94, "Home A mF1": np.nan, "Home B wF1": 0.75, "Home B mF1": np.nan})
    return pd.DataFrame(rows).round(3)

def table2():
    rows = []
    for (key, m, mode), r in RESULTS.items():
        rows.append({"Dataset": key.replace("_", " "), "Model": m, "Evaluation Setting": "Zero-shot" if mode == "zero" else f"Few-shot (K={CFG['K_SHOT']})",
                     "Weighted F1": round(r["wf1"], 3), "95% Bootstrap CI": f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]", "Macro F1": round(r["mf1"], 3),
                     "Test windows": r["n_windows"], "Test instances": r["n_instances"]})
    return pd.DataFrame(rows)

t1, t2 = table1(), table2()
print("Table 1: UCI ADL recognition (the published GPT-3.5 row is context only, not a controlled comparison)"); display(t1)
print("Table 2: weighted F1 with 95% activity-instance cluster-bootstrap CIs"); display(t2)
t1.to_csv(OUT_DIR / "tables/table1_uci.csv", index=False); t2.to_csv(OUT_DIR / "tables/table2_ci.csv", index=False)

In [ ]:
# Few-shot effect on identical held-out MARBLE windows (Section 4.3, Figure 4)
fs = pd.DataFrame([{"Model": m, "Zero-shot wF1": RESULTS[("MARBLE", m, "zero")]["wf1"], "Few-shot wF1": RESULTS[("MARBLE", m, "few")]["wf1"]}
                   for m in CFG["MODELS"]])
fs["Delta (points)"] = (fs["Few-shot wF1"] - fs["Zero-shot wF1"]) * 100
display(fs.round(3))

# Output-parser statistics per run (Supplementary table on parser fallbacks, Section 3.6)
parse_tbl = pd.DataFrame([{"Dataset": k[0], "Model": k[1], "Mode": k[2], **{m: r["parse"].get(m, 0) for m in ("brace", "substring", "stem", "ambiguous", "invalid")}}
                          for k, r in RESULTS.items()])
display(parse_tbl)
parse_tbl.to_csv(OUT_DIR / "tables/parser_statistics.csv", index=False)

### 11.2 Repeatability of greedy decoding (Section 3.11)

In [ ]:
rep_rows = [dict(model=m, dataset="UCI_A", **repeatability_check(prepare("UCI_A"), m, "zero")) for m in CFG["MODELS"]]
rep = pd.DataFrame(rep_rows); display(rep); rep.to_csv(OUT_DIR / "tables/repeatability.csv", index=False)

### 11.3 Privacy: live egress capture + cloud counterfactual (Section 4.4)
Runs a fresh benchmark pass (cache bypassed on a sample) under the egress monitor. Pull all models **before** this cell.

In [ ]:
prep_m = prepare("MARBLE")
probe = prep_m.test.text.drop_duplicates().head(60).tolist()
with EgressMonitor() as mon:
    backend = make_backend(CFG["ASSIST_MODEL"]); sysmsg = system_prompt(prep_m.ds, prep_m.ds.classes)
    for t in probe: backend(sysmsg, zero_shot_user(t))
egress = mon.report(); print(json.dumps(egress, indent=2, default=str))
print("\nInference-related external connections observed:", egress["n_external_connections"])

cf = cloud_counterfactual(prep_m, mode="zero")
display(pd.Series(cf, name="cloud counterfactual (MARBLE cadence)").to_frame())
json.dump(dict(egress=egress, cloud=cf), open(OUT_DIR / "tables/privacy.json", "w"), indent=2, default=str)

### 11.4 Energy per inference and the 70B memory check (Section 4.5)

In [ ]:
energy = pd.DataFrame([measure_energy(prepare("UCI_A"), m) for m in CFG["MODELS"]]).rename(
    columns={"gross_J": "Gross J/inf", "net_J": "Net J/inf", "tok_s": "Throughput (tok/s)"})
display(energy.round(1)); energy.to_csv(OUT_DIR / "tables/table3_energy.csv", index=False)
display(memory_report(CFG["MODELS"]).round(1))

In [ ]:
# Optional: actually try to load the 70B / 5-bit model (pull it first; skipped in demo mode). Expect an OOM / failure on 128 GB unified memory.
def try_big_model():
    if DEMO_MODE: return print("demo mode: skipped")
    try:
        ollama_generate(CFG["BIG_MODEL_TAG"], "Answer with one word.", "Hello")
        print("70B loaded and answered; inspect /api/ps for resident size.")
    except Exception as e:
        print("70B could not be served on this machine:", repr(e)[:300])
    display(memory_report({}).round(1))
# try_big_model()    # uncomment after `ollama pull llama3.1:70b-instruct-q5_K_M`

### 11.5 Assistive automation on the recognised streams (Sections 4.6, 4.7)

In [ ]:
def assistive_evaluation(key):
    prep = prepare(key); full = prep.win.copy()
    preds, _ = predict_windows(prep, CFG["ASSIST_MODEL"], "zero", frame=full)       # recognise the *whole* stream (zero-shot)
    full["pred"] = preds
    W, S = prep.ds.win, prep.ds.step
    eps_p = build_episodes(full, "pred", W, S); eps_t = build_episodes(full.assign(pred=full.label), "pred", W, S)
    act_p, act_t = run_rules(eps_p), run_rules(eps_t)
    days = (full.we.max() - full.ws.min()) / 86400
    return dict(key=key, days=days, episodes=eps_p, edf=episodes_df(eps_p), actions=act_p, truth_actions=act_t, win=full,
                per_profile=Counter(a.profile for a in act_p), caregiver_flagged=sum(a.human_review for a in act_p), match=match_actions(act_p, act_t))

ASSIST = {k: assistive_evaluation(k) for k in ("UCI_A", "UCI_B")}
for k, r in ASSIST.items():
    print(f"{k}: {len(r['actions'])} actions over {r['days']:.1f} days | per profile {dict(r['per_profile'])} | caregiver-flagged (never auto-sent): {r['caregiver_flagged']}")
display(audit_log(ASSIST["UCI_A"]["actions"]).head(15))

scen_df, scen_sum = run_scenarios()
display(scen_df); print("Scenario verification:", scen_sum)

t4 = pd.DataFrame([dict(Home=k.replace("UCI_", "Home "), **{c: r["match"][c] for c in ("total_actions", "recognition_error_actions", "correct_recognition_actions", "error_triggered_rate", "missed_actions")})
                   for k, r in ASSIST.items()])
tot = t4[["total_actions", "recognition_error_actions", "correct_recognition_actions", "missed_actions"]].sum()
t4.loc[len(t4)] = ["Overall", *tot[["total_actions", "recognition_error_actions", "correct_recognition_actions"]],
                   tot["recognition_error_actions"] / max(1, tot["total_actions"]), tot["missed_actions"]]
t4["error_triggered_rate"] = (t4["error_triggered_rate"] * 100).round(1).astype(str) + "%"
print("Table 4: sensitivity of assistive actions to recognition errors"); display(t4)
t4.to_csv(OUT_DIR / "tables/table4_error_propagation.csv", index=False)

rob = robustness_sweep(ASSIST["UCI_A"]["win"], prepare("UCI_A").ds.classes, prepare("UCI_A").ds.win, prepare("UCI_A").ds.step)
print("Controlled robustness sweep (Home A)"); display(rob.round(3)); rob.to_csv(OUT_DIR / "tables/robustness_sweep.csv", index=False)

### 11.6 Behavioral-change detection (Section 4.8)

In [ ]:
beh = {}
for k, r in ASSIST.items():
    beh[k] = behaviour_experiment(r["edf"])
    print(f"\n{k}: days {beh[k]['days']}  | natural flag rate {beh[k]['baseline_flag_rate']:.1%} | injected-perturbation detection {beh[k]['detection']}")
    display(beh[k]["natural"])
t5 = pd.DataFrame([dict(Home=k.replace("UCI_", "Home "), **{"Evaluation days": len(b["days"]), "Naturally flagged days": int(b["natural"].flagged.sum()),
                   "Baseline flag rate": f"{b['baseline_flag_rate']:.1%}"}, **{f"Detect {n}": f"{v:.1%}" for n, v in b["detection"].items()}) for k, b in beh.items()])
print("Table 5: behavioral-change proof of concept (stress test; NOT clinical validation)"); display(t5); t5.to_csv(OUT_DIR / "tables/table5_behaviour.csv", index=False)

## 12. Figures (Figures 2 – 7)

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
MODELS = list(CFG["MODELS"]); COL = dict(zip(MODELS, ["#4C78A8", "#F58518", "#54A24B"]))
def savefig(name): plt.tight_layout(); plt.savefig(OUT_DIR / "figures" / f"{name}.png", dpi=200); plt.show()

# Fig 2: zero-shot weighted F1 on MARBLE
fig, ax = plt.subplots(figsize=(5, 3.4))
v = [RESULTS[("MARBLE", m, "zero")]["wf1"] for m in MODELS]
ci = np.array([RESULTS[("MARBLE", m, "zero")]["ci"] for m in MODELS])
ax.bar(MODELS, v, color=[COL[m] for m in MODELS], yerr=[np.array(v) - ci[:, 0], ci[:, 1] - np.array(v)], capsize=4)
for i, x in enumerate(v): ax.text(i, x + 0.02, f"{x:.3f}", ha="center")
ax.set_ylim(0, 1); ax.set_ylabel("Weighted F1"); ax.set_title("MARBLE zero-shot"); savefig("fig2_marble_zero_shot")

# Fig 3: per-class F1 on MARBLE (zero-shot)
cls = prepare("MARBLE").classes; x = np.arange(len(cls)); w = 0.27
fig, ax = plt.subplots(figsize=(8, 3.6))
for j, m in enumerate(MODELS): ax.bar(x + (j - 1) * w, [RESULTS[("MARBLE", m, "zero")]["per_class"][c] for c in cls], w, label=m, color=COL[m])
ax.set_xticks(x); ax.set_xticklabels(cls, rotation=30, ha="right"); ax.set_ylabel("F1"); ax.set_title("MARBLE per-class F1 (zero-shot)"); ax.legend(frameon=False)
savefig("fig3_marble_per_class")

# Fig 4: zero-shot vs few-shot
fig, ax = plt.subplots(figsize=(5.5, 3.6)); x = np.arange(len(MODELS))
ax.bar(x - 0.18, fs["Zero-shot wF1"], 0.36, label="Zero-shot", color="#9ecae1"); ax.bar(x + 0.18, fs["Few-shot wF1"], 0.36, label=f"Few-shot (K={CFG['K_SHOT']})", color="#3182bd")
for i, d in enumerate(fs["Delta (points)"]): ax.text(i, max(fs.iloc[i, 1:3]) + 0.02, f"{d:+.1f} pts", ha="center")
ax.set_xticks(x); ax.set_xticklabels(MODELS); ax.set_ylim(0, 1); ax.set_ylabel("Weighted F1"); ax.legend(frameon=False); ax.set_title("Few-shot effect on identical MARBLE test windows")
savefig("fig4_few_shot_effect")

# Fig 5: privacy / cost contrast
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, (lab, c, z) in zip(axs, [("API calls / day", cf["api_calls_per_day"], 0), ("Data egress (MB / day)", cf["activity_text_MB_per_day"], 0), ("Annual cost (USD)", cf["usd_per_year"], 0)]):
    ax.bar(["Cloud", "On-device"], [c, z], color=["#e45756", "#54a24b"]); ax.set_title(lab); ax.text(0, c, f"{c:,.1f}", ha="center", va="bottom"); ax.text(1, 0, "0", ha="center", va="bottom")
savefig("fig5_privacy_cost")

# Fig 6: accuracy-energy trade-off
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for _, r in energy.iterrows():
    ax.scatter(r["Net J/inf"], RESULTS[("UCI_A", r["model"], "zero")]["wf1"], s=90, color=COL[r["model"]]); ax.annotate(r["model"], (r["Net J/inf"], RESULTS[("UCI_A", r["model"], "zero")]["wf1"]), textcoords="offset points", xytext=(6, 4))
ax.set_xlabel("Net energy per inference (J)"); ax.set_ylabel("UCI Home A weighted F1"); ax.set_title("Accuracy-energy trade-off"); savefig("fig6_accuracy_energy")

# Fig 7: assistive actions per profile
fig, ax = plt.subplots(figsize=(5.5, 3.5)); x = np.arange(len(PROFILES))
for j, (k, r) in enumerate(ASSIST.items()): ax.bar(x + (j - 0.5) * 0.36, [r["per_profile"].get(p, 0) for p in PROFILES], 0.36, label=k.replace("UCI_", "Home "))
ax.set_xticks(x); ax.set_xticklabels([p.capitalize() for p in PROFILES]); ax.set_ylabel("Actions after de-bouncing"); ax.legend(frameon=False); ax.set_title("Assistive actions per disability profile")
savefig("fig7_assistive_actions")

## 13. Export a machine-readable summary

In [ ]:
summary = dict(
    demo_mode=DEMO_MODE, config={k: v for k, v in CFG.items() if k != "MODELS"}, models=CFG["MODELS"],
    recognition={f"{k[0]}|{k[1]}|{k[2]}": dict(wf1=r["wf1"], mf1=r["mf1"], ci95=r["ci"], n_windows=r["n_windows"], n_instances=r["n_instances"], parser=r["parse"]) for k, r in RESULTS.items()},
    privacy=dict(egress=egress, cloud_counterfactual=cf), energy=energy.to_dict("records"),
    assistive=dict(scenarios=scen_sum, actions={k: len(r["actions"]) for k, r in ASSIST.items()}, per_profile={k: dict(r["per_profile"]) for k, r in ASSIST.items()}),
    behaviour={k: dict(days=b["days"], baseline_flag_rate=b["baseline_flag_rate"], detection=b["detection"]) for k, b in beh.items()})
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Saved tables, figures and summary.json to", OUT_DIR.resolve())
if DEMO_MODE: print("\n*** DEMO_MODE: synthetic data + mock LLM. These numbers are NOT results. Set DEMO_MODE=False for the real experiment. ***")